# Silver Layer - Data Cleansing and Transformation

This notebook reads from bronze table and applies:
- Data quality checks (remove nulls)
- Standardization (trim whitespace, uppercase state codes)
- Add business metadata

In [0]:
# Get environment parameters from job parameters or dbutils widgets
try:
    catalog = dbutils.widgets.get("catalog")
    schema = dbutils.widgets.get("schema")
except Exception:
    # Fallback for local development
    catalog = "gitflow"
    schema = "gitflow_dev"

print(f"Target Catalog: {catalog}")
print(f"Target Schema: {schema}")

## Read from Bronze Table

In [0]:
df_bronze = spark.table(f"{catalog}.{schema}.bronze_customers")
print(f"Bronze records: {df_bronze.count()}")
df_bronze.display()

## Apply Data Quality Rules

In [0]:
from pyspark.sql.functions import col, trim, upper, current_timestamp, current_date

# Data quality transformations
df_silver = (
    df_bronze.filter(col("customer_id").isNotNull())
    .filter(col("customer_name").isNotNull())
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("state", upper(trim(col("state"))))
    .withColumn("processed_timestamp", current_timestamp())
    .withColumn("processing_date", current_date())
)

print(f"Silver records after quality checks: {df_silver.count()}")
df_silver.display()

## Write to Silver Table

In [0]:
table_name = f"{catalog}.{schema}.silver_customers"
df_silver.write.mode("overwrite").saveAsTable(table_name)

print(f"✅ Silver table created: {table_name}")
print(f"Records written: {df_silver.count()}")

## Verify Silver Table

In [0]:
spark.sql(f"SELECT * FROM {catalog}.{schema}.silver_customers LIMIT 10").display()